# Molecular Representations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChemAI-Lab/AI4Chem/blob/main/website/modules/05-molecular_representations.ipynb)

In [ ]:
# ! pip install py3dmol rdkit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import py3Dmol
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw, rdMolDescriptors, rdDistGeom, rdMolTransforms, QED
from rdkit.Chem.Scaffolds.MurckoScaffold import GetScaffoldForMol
from rdkit.Chem.rdmolops import GetAdjacencyMatrix
from rdkit.Chem.Draw import IPythonConsole, SimilarityMaps
from rdkit.Chem import Descriptors
from rdkit import Chem
from rdkit.Chem import Draw

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

import networkx as nx

IPythonConsole.ipython_useSVG = True
IPythonConsole.drawOptions.addAtomIndices = True
IPythonConsole.molSize = 300, 300

# Datasets

You can download the data from [link](https://moleculenet.org/datasets-1)
1. **Lipophilicity dataset**: Experimental results of octanol/water distribution coefficient(logD at pH 7.4).
2. **HIV**: Experimentally measured abilities to inhibit HIV replication.
3. **BBBP**: Binary labels of blood-brain barrier penetration(permeability).

In [ ]:
# load datasets
# Lipophilicity dataset
lipo_data_file = 'https://raw.githubusercontent.com/ChemAI-Lab/AI4Chem/main/website/modules/data/Lipophilicity.csv'
lipo_data = pd.read_csv(lipo_data_file)
lipo_data.head()

smiles_lipo = lipo_data['smiles'].tolist()
logp_values = np.array(lipo_data['exp'].tolist())
print(f'Number of molecules in Lipophilicity dataset: {len(smiles_lipo)}')

plt.hist(logp_values, bins=30)
plt.xlabel('LogP values')
plt.ylabel('Frequency')
plt.title('Distribution of LogP values in Lipophilicity dataset')
plt.show()


In [ ]:
# HIV dataset
hiv_data_file = 'https://raw.githubusercontent.com/ChemAI-Lab/AI4Chem/main/website/modules/data/HIV.csv'
hiv_data = pd.read_csv(hiv_data_file)
hiv_data.head()

smiles_hiv = hiv_data['smiles'].tolist()
activity_values = np.array(hiv_data['HIV_active'].tolist())


plt.bar(['Inactive', 'Active'], [np.sum(activity_values == 0), np.sum(activity_values == 1)])
plt.ylabel('Number of Molecules')
plt.title('Distribution of Activity in HIV dataset')
plt.show()

In [ ]:
# BBBP dataset
bbbp_data_file = 'https://raw.githubusercontent.com/ChemAI-Lab/AI4Chem/main/website/modules/data/BBBP.csv'
bbbp_data = pd.read_csv(bbbp_data_file)
bbbp_data.head()

smiles_bbbp = bbbp_data['smiles'].tolist()
bbbp_values = np.array(bbbp_data['p_np'].tolist())

# Binary labels of blood-brain barrier penetration(permeability).
plt.bar(['Inactive', 'Active'], [
        np.sum(bbbp_values == 0), np.sum(bbbp_values == 1)])
plt.ylabel('Number of Molecules')
plt.title('Distribution of Activity in bbbp dataset')
plt.show()

# How can we represent molecules in a computer?

## Molecular descriptors

*Molecular descriptors capture diverse parts of the structural information of molecules and they are the support of many contemporary computer-assisted toxicological and chemical applications.*

Being able to numerically represent molecules aid with the prediction problem,
$$
y = f(\mathbf{{\cal M}})= f_\theta(\phi_\theta(\mathbf{{\cal M}})),
$$
where $\phi_\theta(\mathbf{{\cal M}})$ is the molecular descriptors used for supervised learning tasks like regression or classification.

<img src="https://raw.github.com/RodrigoAVargasHdz/CHEM-4PB3/master/Course_Notes/Figures/440801_1_En_1_Fig1_HTML.png"  width="500" height="700">

Figure from [link](https://link.springer.com/protocol/10.1007/978-1-4939-7899-1_1)


Molecular descriptors can describe different levels of information, from bulk properties to complex three-dimensional definitions or substructure frequency.
1. **0-Dimensional**:\
   atom counts (e.g., number of carbon atoms), molecular weight, and sum or average of atomic properties (e.g., atomic van der Waals volumes).
2. **1-Dimensional**:\
   molecules are perceived as a set of substructures, such as functional groups or atom-centered fragments.
3. **2-Dimensional**:\
   molecule is represented as a graph, whose vertexes are the atoms and edges are the bonds, and specific chemical properties of atoms are considered.
4. **3-Dimensional**:\
   descriptors deriving from 3D representation.
5. **4-Dimensional**:\
   molecular geometry combined with an “additional dimension/information”, e.g., representing each ligand by an ensemble of conformations, protonation states, and/or orientations.


### 0-Dimensional: Classical molecular descriptors and binary fingerprints ###


**Classical molecular descriptors** (MDs) are designed to encode a precise structural/chemical feature (or a set of features of different complexity) into one, single number. <br>
They can also be combined with other molecular properties that could be efficiently estimated. 

**RDKit molecular descriptors** ([list](https://www.rdkit.org/docs/source/rdkit.Chem.rdMolDescriptors.html?highlight=calcnumhba#rdkit.Chem.rdMolDescriptors.CalcNumHBA))


In [ ]:
def compute_classical_descriptors(smiles, descriptor_names=None):
    """
    Return (values_array, names_list) of RDKit classical descriptors for a SMILES string.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES: %s" % smiles)

    all_desc = Descriptors._descList  # list of (name, function)
    if descriptor_names is None:
        desc_list = all_desc
    else:
        wanted = set(descriptor_names)
        desc_list = [(n, f) for n, f in all_desc if n in wanted]

    names = [n for n, _ in desc_list]
    values = []
    for n, f in desc_list:
        try:
            values.append(float(f(mol)))
        except Exception:
            values.append(np.nan)

    return np.array(values, dtype=float), names


# Example usage with caffeine_smiles already defined in the notebook
caffeine_smiles = 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'

vec, desc_names = compute_classical_descriptors(caffeine_smiles)
print("Computed %d descriptors" % len(desc_names))
for n, v in zip(desc_names[:], vec[:]):  # show first 20
    print(n, f"{v:.2f}")

In [ ]:
import numpy as np
descriptor_values = []
for smi in smiles_lipo:
    vec, _ = compute_classical_descriptors(smi)
    descriptor_values.append(vec)

X_lipo = np.array(descriptor_values)
y_lipo = logp_values

# Check for NaNs in X_lipo before splitting
nan_mask = np.isnan(X_lipo)
n_nans = nan_mask.sum()
print("Total NaNs in X_lipo:", n_nans)

if n_nans > 0:
    # Which columns have NaNs and how many
    col_nan_counts = nan_mask.sum(axis=0)
    nan_cols = np.where(col_nan_counts > 0)[0]
    print("Columns with NaNs:", nan_cols)
    print("NaNs per column:", col_nan_counts[nan_cols])

col_keep_mask = ~np.isnan(X_lipo).any(axis=0)
dropped = np.where(~col_keep_mask)[0]
print(f'Dropping {len(dropped)} columns with NaNs')

X_lipo = X_lipo[:, col_keep_mask]
desc_names = [d for d, keep in zip(desc_names, col_keep_mask) if keep]


X_lipo_train, X_lipo_test, y_lipo_train, y_lipo_test = train_test_split(X_lipo, y_lipo, test_size=0.2, random_state=42)
print(f'Training set size: {X_lipo_train.shape[0]} molecules')
print(f'Test set size: {X_lipo_test.shape[0]} molecules')

In [ ]:
# Mosaic of histograms for descriptors (subset)
import math
import matplotlib.pyplot as plt

# Topological Polar Surface Area (TPSA)
desc_names_subset = ['MolWt', 'ExactMolWt', 'NumRotatableBonds',  'NumAromaticRings',
                     'HeavyAtomCount', 'NumHDonors', 'NumHAcceptors', 'TPSA', "NumValenceElectrons"]

n = len(desc_names_subset)
cols = 3  # adjust as needed
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 3.5), constrained_layout=True)
axes = axes.flatten() if n > 1 else [axes]

for i, desc_name in enumerate(desc_names_subset):
    ax = axes[i]
    idx = desc_names.index(desc_name)
    ax.hist(X_lipo[:, idx], bins=30, color='skyblue', edgecolor='black')
    ax.set_title(f'Distribution of {desc_name}')
    ax.set_xlabel(desc_name)
    ax.set_ylabel('Frequency')
    ax.grid(axis='y', alpha=0.75)

# Hide any unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.show()

In [ ]:
# normalize data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# Doesn't affect the results
y_lipo_train_norm = scaler.fit_transform(y_lipo_train.reshape(-1, 1)).flatten()
y_lipo_test_norm = scaler.transform(y_lipo_test.reshape(-1, 1)).flatten()

# Without normalization linear model does not work well
X_lipo_train_norm = scaler.fit_transform(X_lipo_train)
X_lipo_test_norm = scaler.transform(X_lipo_test)

In [ ]:
# Linear Regression
lr = LinearRegression(fit_intercept=True)
lr.fit(X_lipo_train_norm, y_lipo_train)
print("Linear model Coefficients:")
print(lr.coef_)

y_lipo_pred_train = lr.predict(X_lipo_train_norm)
mae_tr, r2_train = mean_absolute_error(y_lipo_train, y_lipo_pred_train), r2_score(y_lipo_train, y_lipo_pred_train)
y_lipo_pred_test = lr.predict(X_lipo_test_norm)
mae_test, r2_test = mean_absolute_error(y_lipo_test, y_lipo_pred_test), r2_score(y_lipo_test, y_lipo_pred_test)

print(f"Training MAE: {mae_tr:.3f}, R2: {r2_train:.3f}")
print(f"Test MAE: {mae_test:.3f}, R2: {r2_test:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

# Train panel
axes[0].scatter(y_lipo_train, y_lipo_pred_train, alpha=0.6)
axes[0].plot([min(y_lipo_train), max(y_lipo_train)], [
             min(y_lipo_train), max(y_lipo_train)], color='red', linestyle='--')
axes[0].text(0.05, 0.9, f'MAE: {mae_tr:.3f}\nR2: {r2_train:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
axes[0].set_xlabel('True LogP')
axes[0].set_ylabel('Predicted LogP')
axes[0].set_title('Train')

# Test panel
axes[1].scatter(y_lipo_test, y_lipo_pred_test, alpha=0.6)
axes[1].plot([min(y_lipo_test), max(y_lipo_test)], [
             min(y_lipo_test), max(y_lipo_test)], color='red', linestyle='--')
axes[1].text(0.05, 0.9, f'MAE: {mae_test:.3f}\nR2: {r2_test:.3f}',
             transform=axes[1].transAxes, fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
axes[1].set_xlabel('True LogP')
axes[1].set_ylabel('Predicted LogP')
axes[1].set_title('Test')

plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


def get_device():
    """
    Automatically selects the best available device (CUDA, MPS, or CPU).
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


def torch_to_numpy(tensor):
    return np.array(tensor.tolist())


device = get_device()
print(f"Using device: {device}")

In [ ]:
# Neural Network Regression
class SimpleNNS(nn.Module):
    def __init__(self, input_size=2, hidden_size=2, output_size=1, n_layers=3, activation=nn.Tanh):
        super().__init__()
        layers = [nn.Linear(input_size, hidden_size), activation()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_size, hidden_size), activation()]
        layers.append(nn.Linear(hidden_size, output_size))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
    
# Prepare data for PyTorch
X_lipo_train_tensor = torch.tensor(X_lipo_train_norm, dtype=torch.float32)
y_lipo_train_tensor = torch.tensor(y_lipo_train.reshape(-1, 1), dtype=torch.float32)
X_lipo_test_tensor = torch.tensor(X_lipo_test_norm, dtype=torch.float32)
y_lipo_test_tensor = torch.tensor(y_lipo_test.reshape(-1, 1), dtype=torch.float32)

n_epochs = 500
batch_size = 64
learning_rate = 0.001
train_dataset = TensorDataset(X_lipo_train_tensor, y_lipo_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
model = SimpleNNS(input_size=X_lipo_train.shape[1], 
                  hidden_size=512, output_size=1,
                  n_layers=3, activation=nn.LeakyReLU).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X_batch.size(0)
    epoch_loss /= len(train_loader.dataset)
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch + 1}/{n_epochs}], Loss: {epoch_loss:.4f}')

In [ ]:
y_lipo_train_pred = model(X_lipo_train_tensor.to(device)).detach().cpu()
y_lipo_train_pred = torch_to_numpy(y_lipo_train_pred).flatten()
y_lipo_test_pred = model(X_lipo_test_tensor.to(device)).detach().cpu()
y_lipo_test_pred = torch_to_numpy(y_lipo_test_pred).flatten()

mae_tr, r2_train = mean_absolute_error(y_lipo_train, y_lipo_train_pred), r2_score(y_lipo_train, y_lipo_train_pred)
mae_test, r2_test = mean_absolute_error(y_lipo_test, y_lipo_test_pred), r2_score(y_lipo_test, y_lipo_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

# Train panel
axes[0].scatter(y_lipo_train, y_lipo_train_pred, alpha=0.6)
axes[0].plot([min(y_lipo_train), max(y_lipo_train)], [
             min(y_lipo_train), max(y_lipo_train)], color='red', linestyle='--')
axes[0].text(0.05, 0.9, f'MAE: {mae_tr:.3f}\nR2: {r2_train:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
axes[0].set_xlabel('True LogP')
axes[0].set_ylabel('Predicted LogP')
axes[0].set_title('Train')

# Test panel
axes[1].scatter(y_lipo_test, y_lipo_test_pred, alpha=0.6)
axes[1].plot([min(y_lipo_test), max(y_lipo_test)], [
             min(y_lipo_test), max(y_lipo_test)], color='red', linestyle='--')
axes[1].text(0.05, 0.9, f'MAE: {mae_test:.3f}\nR2: {r2_test:.3f}',
             transform=axes[1].transAxes, fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
axes[1].set_xlabel('True LogP')
axes[1].set_ylabel('Predicted LogP')
axes[1].set_title('Test')

plt.show()

# Molecules as Strings

## SMILES
The **Simplified Molecular Input Line Entry System (SMILES)** is a notation that allows a user to represent a chemical structure in a way that can be used by computer programs. <br>
It is a compact and linear way of describing the structure of molecules.

<br>

### Breaking Down the SMILES String

1. **Atoms and Bonds**: In SMILES, atoms are represented by their chemical symbols (e.g., C for carbon, N for nitrogen, O for oxygen). <br>
   Upper case letters means non-aromatic atoms, and lower case letters aromatic atoms. <br>
   Bonds between atoms are often implied but can also be explicitly represented (e.g., '=' for a double bond). <br>

2. **Ring Structures**: Numbers in the SMILES string are used to denote ring closures. In caffeine, the numbers 1 and 2 indicate the points at which ring structures are formed. <br>
   For example, `N1C=NC2` indicates that the nitrogen atom (N) is involved in two ring structures, connecting at points labeled 1 and 2.

3. **Branches**: Parentheses are used to indicate branches from the main chain. In the case of caffeine, `(=O)` and `(C(=O)N2C)` are branches off the main structure.

In [ ]:
caffeine_smiles = 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'

# plot molecule in rdkit
caffeine_mol = Chem.MolFromSmiles(caffeine_smiles)
caffeine_mol

In [ ]:
smiles = "Cn1cnc2n(C)c(=O)n(C)c(=O)c12"  # caffeine (one valid SMILES)
mol = Chem.MolFromSmiles(smiles)

# Define substructures (SMARTS) and colors (RGB in 0..1)
patterns = [
    # ("Imide carbonyls", "Cn", (1.0, 0.2, 0.2)),   # reddish
    # ("N-methyl",        "n1cncc1",  (0.2, 0.85, .3)),    # bluish
    # ("C=O",              "c=O",    (0.9, 0.6, 0.0)),   # orange
    ("methyl",  "C",     (0.2, 0.85, 0.3)),   # greenish
    ("NCN",  "ncn",     (0.2, 0.85, 0.3)),   # greenish
]

highlight_atoms = set()
highlight_bonds = set()
atom_colors = {}
bond_colors = {}

for name, smarts, color in patterns:
    q = Chem.MolFromSmarts(smarts)
    for match in mol.GetSubstructMatches(q):
        # color atoms
        for a in match:
            highlight_atoms.add(a)
            atom_colors[a] = color

        # color bonds internal to the match
        for i in range(len(match)):
            for j in range(i + 1, len(match)):
                b = mol.GetBondBetweenAtoms(match[i], match[j])
                if b is not None:
                    bi = b.GetIdx()
                    highlight_bonds.add(bi)
                    bond_colors[bi] = color

# Ensure highlighted lists align with color dict keys (avoid default red)
highlight_atoms = list(atom_colors.keys())
highlight_bonds = list(bond_colors.keys())

img = Draw.MolToImage(
    mol,
    size=(450, 350),
    highlightAtoms=highlight_atoms,
    highlightBonds=highlight_bonds,
    highlightAtomColors=atom_colors,
    highlightBondColors=bond_colors,
    highlightBondWidthMultiplier=18,  # makes colors pop
)
img


<!DOCTYPE html>
<html>
<head>
    <style>
        .centered-image {
            display: block;
            margin-left: auto;
            margin-right: auto;
            width: 50%;
        }
    </style>
</head>
<body>

<a href="https://www.chemistryworld.com/opinion/weiningers-smiles/4014639.article" target="_blank">
    <img src="https://d2cbg94ubxgsnp.cloudfront.net/Pictures/1024x536/7/1/6/516716_smiles_54413.png"
         alt="Varied Initial Conditions for Gradient Descent"
         class="centered-image">
</a>

<br>
<figcaption align = "center"><b>Figure 1 - SMILES naming convention. Ciprofloxacin written as a Smiles string. Figure from
Andrea Sella.</b></figcaption>

</body>
</html>

In [ ]:
# one-hot-encoding
# ALPHABET: define SMILES characters

max_len = 30
SMILES_CHARS = ["7", "6", "o", "]", "3", "s", "(", "-", "S", "/", "B", "4", "[", ")", "#", "I",
                "l", "O", "H", "c", "1", "@", "=", "n", "P", "8", "C", "2", "F", "5", "r", "N", "+", "\\", " "]
# index
smi2index = dict((c, i) for i, c in enumerate(SMILES_CHARS))


def smiles_to_one_hot(smiles, maxlen=max_len):
    X = np.zeros((maxlen, len(SMILES_CHARS)))  # (maxlen, dictionary)
    # print(smiles,type(smiles))
    smiles = smiles.replace('\n', '')
    for i, c in enumerate(smiles):
        X[i, smi2index[c]] = 1
    return X


# caffeine one hot
caffeine_smiles = 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'
print(caffeine_smiles.split())

caffeine_one_hot = smiles_to_one_hot(caffeine_smiles)

print(caffeine_one_hot.shape)  # (120, 56)


plt.figure(figsize=(10, 10))
plt.imshow(caffeine_one_hot.T, cmap='binary')
# plt.xlabel('Tokens')
# plt.ylabel('SMILES')


caffeine_smiles_pad = caffeine_smiles + " " * (max_len - len(caffeine_smiles))

plt.title('One-hot encoding for %s' % caffeine_smiles)
plt.xticks(np.arange(len(list(caffeine_smiles))),
           list(caffeine_smiles), fontsize=8)
plt.yticks(np.arange(len(list(SMILES_CHARS))),
           list(SMILES_CHARS), fontsize=8)
plt.xlabel('Tokens')
plt.ylabel('SMILES Dictionary')

In [ ]:
# get tokens from a SMILES dataset
# Use the BBBP dataset SMILES to build a unique token dictionary
import re

# Minimal RDKit-aware SMILES tokenizer
_smiles_token_re = re.compile(
    r"(\[[^\]]+\])|Br|Cl|@@?|%\d{2}|\d|=|#|-|\\|/|:|\.|\(|\)|[A-Za-z]|\*"
)

def tokenize_smiles(smiles):
    if not isinstance(smiles, str):
        return []
    # use finditer to avoid empty-string matches from capture groups
    return [m.group(0) for m in _smiles_token_re.finditer(smiles)]

# Build token set from BBBP SMILES
tokens = set()
for s in smiles_bbbp:
    for tok in tokenize_smiles(s):
        tokens.add(tok)

# Reserve 0 for PAD and 1 for UNK
token_dict = {'<PAD>': 0, '<UNK>': 1}
for i, tok in enumerate(sorted(tokens), start=2):
    token_dict[tok] = i

print(f'Num tokens (incl. PAD/UNK): {len(token_dict)}')


In [ ]:
# Tokenize SMILES for a CNN (binary classification, imbalanced)

# Encode each SMILES into a list of token ids
def encode_smiles(smiles_list, token_dict, unk_token='<UNK>'):
    if unk_token not in token_dict:
        token_dict = dict(token_dict)
        token_dict[unk_token] = len(token_dict)
    sequences = []
    for s in smiles_list:
        toks = tokenize_smiles(s)
        seq = [token_dict.get(t, token_dict[unk_token]) for t in toks]
        sequences.append(seq)
    return sequences, token_dict

In [ ]:
caffeine_smiles = 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'  # smiles_bbbp[1000]
caffeine_toks = tokenize_smiles(caffeine_smiles)
print(smiles_bbbp[1000])
print("tokens:", caffeine_toks)
print("token_dict size:", len(token_dict))
print("missing:", [t for t in caffeine_toks if t not in token_dict])
print("UNK index:", token_dict.get("<UNK>", None))

In [ ]:
sequences, token_dict = encode_smiles(smiles_bbbp, token_dict)


# Distribution of token counts per SMILES
token_counts = [len(seq) for seq in sequences]
plt.figure(figsize=(6, 4))
plt.hist(token_counts, bins=50, color='skyblue', edgecolor='black')
plt.title('Token Count Distribution (BBBP SMILES)')
plt.xlabel('Number of tokens')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()


In [ ]:
# Pad sequences to a fixed length for CNN input
max_len = 75
X_bbbp_seq = np.zeros((len(sequences), max_len), dtype=np.int32)
for i, seq in enumerate(sequences):
    X_bbbp_seq[i, :min(len(seq), max_len)] = seq[:max_len]

print('Sequence tensor shape:', X_bbbp_seq.shape)

In [ ]:
# MLP with embedding for BBP classification
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# Labels (keep original: 1 = majority for BBBP)
y_bbbp = np.asarray(bbbp_values).astype(np.float32)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_bbbp_seq, y_bbbp, test_size=0.2, random_state=42, stratify=y_bbbp
)

# Torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256)

vocab_size = len(token_dict)
embed_dim = 64
hidden_dim = 128
n_layers = 3

class SimpleNNwEmb(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers=3, activation=nn.Tanh, with_sigmoid=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        layers = [nn.Linear(embed_dim, hidden_dim), activation()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), activation()]
        layers.append(nn.Linear(hidden_dim, 1))
        if with_sigmoid:
            layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        e = self.emb(x)
        e = e.mean(dim=1)
        return self.net(e).squeeze(-1)

model = SimpleNNwEmb(vocab_size, embed_dim, hidden_dim, n_layers=n_layers, activation=nn.Tanh, with_sigmoid=False)

# Sample-weighted BCE to up-weight the minority class (class 0)
pos = y_train.sum()
neg = len(y_train) - pos
# Weight class 0 higher (minority)
w0 = (pos / max(neg, 1.0))
w1 = 1.0
bce = nn.BCEWithLogitsLoss(reduction='none')

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def weighted_loss(logits, y):
    w0_t = y.new_full((), float(w0))
    w1_t = y.new_full((), float(w1))
    weights = torch.where(y >= 0.5, w1_t, w0_t)
    return (bce(logits, y) * weights).mean()

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_y = []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            loss = weighted_loss(logits, yb)
            total_loss += loss.item() * len(xb)
            probs = torch.sigmoid(logits)
            all_probs.append(torch_to_numpy(probs.cpu()))
            all_y.append(torch_to_numpy(yb.cpu()))
    all_probs = np.concatenate(all_probs)
    all_y = np.concatenate(all_y)
    roc = roc_auc_score(all_y, all_probs)
    pr = average_precision_score(all_y, all_probs)
    return total_loss / len(loader.dataset), roc, pr

epochs = 1000
for epoch in range(1, epochs + 1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = weighted_loss(logits, yb)
        loss.backward()
        optimizer.step()
    train_loss, train_roc, train_pr = evaluate(train_loader)
    test_loss, test_roc, test_pr = evaluate(test_loader)
    print(
        f'Epoch {epoch}: train loss={train_loss:.4f}, test loss={test_loss:.4f} | '
        f'train ROC-AUC={train_roc:.3f}, test ROC-AUC={test_roc:.3f} | '
        f'train PR-AUC={train_pr:.3f}, test PR-AUC={test_pr:.3f}'
    )


### Classification report metrics: quick definitions

- **Precision**: Of all samples predicted as positive, how many are truly positive?  
  `precision = TP / (TP + FP)`
- **Recall** (Sensitivity): Of all true positives, how many did we correctly find?  
  `recall = TP / (TP + FN)`
- **F1-score**: Harmonic mean of precision and recall (balances both).  
  `f1 = 2 * (precision * recall) / (precision + recall)`
- **Support**: Number of true samples for each class in the dataset.


In [ ]:
# F1 score and confusion matrix for the trained MLP
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, classification_report

model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    probs = torch.sigmoid(logits)
    y_pred = torch_to_numpy((probs >= 0.5).cpu()).astype(int)
    y_true = torch_to_numpy(y_test_t.cpu()).astype(int)

cm = confusion_matrix(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
print('F1 score:', f1)
print('Confusion matrix:')
print(cm)
print('Classification report:')
print(classification_report(y_true, y_pred, digits=3))


# Convolutional Neural Networks (CNN)

## For sequence data we are going to use 1D CNNs. 

In [ ]:
# 1D CNN for BBBP classification (token sequences)
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# Labels
y_hiv = np.asarray(activity_values).astype(np.float32)

# Train/test split (reuse if already split)
X_train, X_test, y_train, y_test = train_test_split(
    X_bbbp_seq, y_bbbp, test_size=0.2, random_state=42, stratify=y_bbbp
)

# Torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256)

vocab_size = len(token_dict)
embed_dim = 64
num_filters = 128
kernel_size = 5

class CNN1D(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, kernel_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv = nn.Conv1d(embed_dim, num_filters, kernel_size, padding=kernel_size // 2)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(num_filters, 1)

    def forward(self, x):
        # x: (batch, seq_len) -> embed -> (batch, embed_dim, seq_len)
        e = self.emb(x).transpose(1, 2)
        h = self.relu(self.conv(e))
        h = self.pool(h).squeeze(-1)
        return self.fc(h).squeeze(-1)

model = CNN1D(vocab_size, embed_dim, num_filters, kernel_size).to(device)

pos = y_train.sum()
neg = len(y_train) - pos
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_y = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(xb)
            probs = torch.sigmoid(logits)
            all_probs.append(torch_to_numpy(probs.cpu()))
            all_y.append(torch_to_numpy(yb.cpu()))
    all_probs = np.concatenate(all_probs)
    all_y = np.concatenate(all_y)
    roc = roc_auc_score(all_y, all_probs)
    pr = average_precision_score(all_y, all_probs)
    return total_loss / len(loader.dataset), roc, pr

epochs = 100
for epoch in range(1, epochs + 1):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
    train_loss, train_roc, train_pr = evaluate(train_loader)
    test_loss, test_roc, test_pr = evaluate(test_loader)
    print(
        f'Epoch {epoch}: train loss={train_loss:.4f}, test loss={test_loss:.4f} | '
        f'train ROC-AUC={train_roc:.3f}, test ROC-AUC={test_roc:.3f} | '
        f'train PR-AUC={train_pr:.3f}, test PR-AUC={test_pr:.3f}'
    )


In [ ]:
# F1 score and confusion matrix for the trained MLP
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, classification_report

model.eval()
with torch.no_grad():
    logits = model(X_test_t.to(device))
    probs = torch.sigmoid(logits)
    y_pred = torch_to_numpy((probs >= 0.5).cpu()).astype(int)
    y_true = torch_to_numpy(y_test_t.cpu()).astype(int)

cm = confusion_matrix(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
print('F1 score:', f1)
print('Confusion matrix:')
print(cm)
print('Classification report:')
print(classification_report(y_true, y_pred, digits=3))

# ROC-AUC and PR-AUC: quick tutorial



## 1) ROC-AUC (Receiver Operating Characteristic AUC)

- **What it measures:** how well the model ranks positives above negatives across all classification thresholds.

- **Curve axes:**

  - **True Positive Rate (TPR)** = TP / (TP + FN)

  - **False Positive Rate (FPR)** = FP / (FP + TN)

- **Interpretation:**

  - **ROC-AUC = 0.5**: random ranking

  - **ROC-AUC = 1.0**: perfect ranking

- **Strength:** insensitive to class imbalance *in the sense of ranking*.

- **Caveat:** can look optimistic on highly imbalanced data because FPR can stay low even with many false positives.



## 2) PR-AUC (Precision–Recall AUC)

- **What it measures:** how well the model finds positives without flooding with false positives.

- **Curve axes:**

  - **Precision** = TP / (TP + FP)

  - **Recall** = TP / (TP + FN)

- **Interpretation:**

  - **Baseline PR-AUC ≈ positive class fraction** (e.g., 5% positives → baseline ≈ 0.05)

  - Higher is better, 1.0 is perfect.

- **Strength:** much more informative for imbalanced datasets (like many bioactivity tasks).



## 3) Which should I report?

- **Imbalanced data:** prioritize **PR-AUC**, because it reflects how many false positives you create.

- **Balanced data or ranking focus:** ROC-AUC is fine and commonly reported.



## 4) Practical tips

- Track **both** during training.

- Use **PR-AUC** for early stopping on imbalanced datasets.

- Always report the **positive class fraction** so PR-AUC is interpretable.

